# OPSD Household Data — industrial3 processing

Source: Open Power System Data, "Household Data" package (`household_data_60min_singleindex.csv`).
Only the `DE_KN_industrial3` building group meets the ≥10-feature criterion (19 features excl. target).

Every column in this file is a cumulative energy counter (kWh), not an hourly value — confirmed below with `check_cumulative`. Must diff before modeling.

In [ ]:
import sys
sys.path.append(".")
import pandas as pd
from common import report_candidate, check_cumulative

RAW_PATH = "../data/raw/household_data_60min_singleindex.csv"
PROCESSED_PATH = "../data/processed/opsd_household_industrial3.csv"

df = pd.read_csv(RAW_PATH, parse_dates=["utc_timestamp"])
df = df.set_index("utc_timestamp")
df.index.name = "timestamp"
df.shape

In [ ]:
TARGET = "DE_KN_industrial3_grid_import"
ind3_cols = [c for c in df.columns if c.startswith("DE_KN_industrial3_")]
feature_cols = [c for c in ind3_cols if c != TARGET]

raw_ind3 = df[ind3_cols].dropna(how="all")
start, end = raw_ind3.index.min(), raw_ind3.index.max()
raw_ind3 = df.loc[start:end, ind3_cols]
print(f"active window: {start} -> {end}  ({len(raw_ind3)} hourly rows)")

In [ ]:
check_cumulative(raw_ind3, ind3_cols)

## Cumulative -> hourly consumption

Every column is a running meter reading, not a per-hour value. Gaps of at most `MAX_GAP_HOURS` are filled by interpolating the *cumulative* series (defensible for a monotonic counter); larger gaps are left as NaN and dropped after differencing rather than fabricated.

In [ ]:
MAX_GAP_HOURS = 3

interpolated = raw_ind3.interpolate(method="time", limit=MAX_GAP_HOURS)
consumption = interpolated.diff()

before = len(consumption)
consumption = consumption.dropna(how="any")
after = len(consumption)
print(f"dropped {before - after} rows with unresolved gaps (first-diff row + gaps > {MAX_GAP_HOURS}h)")

assert (consumption < 0).sum().sum() == 0, "negative consumption after diff -- investigate before proceeding"
consumption.shape

In [ ]:
report_candidate(consumption, TARGET, feature_cols, freq="1h", name="OPSD industrial3 (processed)")

In [ ]:
consumption.to_csv(PROCESSED_PATH)
print(f"saved: {PROCESSED_PATH}  shape={consumption.shape}")